# 🧠 Axom AI — Fine-tune (Free GPU on Kaggle/Colab)

**Steps:** Run har cell top-to-bottom (Kaggle: 'Run All'). GPU on karo (Settings → Accelerator → GPU T4).

1. Unsloth install
2. Base model load (Llama-3.2-1B)
3. LoRA add
4. Apna data load (data.jsonl upload karo — instruction/output format)
5. Train
6. Test
7. GGUF export → download → Ollama me chalao


## 1. Install Unsloth


In [ ]:
!pip install -q unsloth


## 2. Load base model (small, fits free GPU)


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Llama-3.2-1B-Instruct',
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)


## 3. Add LoRA adapters (only these get trained = fast + light)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)


## 4. Load YOUR data

Upload apni `data.jsonl` (same format as sample_data.jsonl):
`{"instruction": "...", "output": "..."}` — ek line per example.

Kaggle: right panel → '+ Add Input' / Upload. Colab: left folder icon → upload.
Neeche path apne file ke hisaab se badlo.


In [ ]:
import json
from datasets import Dataset

DATA_PATH = 'data.jsonl'   # <-- apni uploaded file ka path yahan do

rows = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
print('Total examples:', len(rows))

def to_chat(ex):
    msgs = [
        {'role':'user','content': ex['instruction']},
        {'role':'assistant','content': ex['output']},
    ]
    text = tokenizer.apply_chat_template(msgs, tokenize=False)
    return {'text': text}

dataset = Dataset.from_list(rows).map(to_chat)
print(dataset[0]['text'][:300])


## 5. Train


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = 'text',
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,          # chhote data ke liye 3-5 theek
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = 'adamw_8bit',
        output_dir = 'outputs',
        seed = 3407,
    ),
)
trainer.train()


## 6. Quick test (trained model se poochho)


In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{'role':'user','content':'Axom AI kya hai?'}]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=128, temperature=0.7)
print(tokenizer.decode(out[0], skip_special_tokens=True))


## 7. Export to GGUF (Ollama-ready) + download


In [ ]:
# q4_k_m = achha balance (chhota + fast). Yeh outputs me .gguf file banayega.
model.save_pretrained_gguf('axom_model', tokenizer, quantization_method = 'q4_k_m')
print('Done! Neeche .gguf file dhundo aur download karo:')
import os
for root,_,files in os.walk('.'):
    for fn in files:
        if fn.endswith('.gguf'):
            print(os.path.join(root, fn))


## 8. Apne PC pe Ollama me chalao

GGUF download karne ke baad, apne PC pe us folder me ek file banao **`Modelfile`** (koi extension nahi):
```
FROM ./axom_model.Q4_K_M.gguf
```
Phir terminal me:
```
ollama create axom -f Modelfile
```
Ab Axom AI project ke `.env` me set karo:
```
OLLAMA_MODEL=axom
```
Django restart → tumhara **apna trained model** chalu! 🎉
